# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

**Workflow (a state machine):** `understand question -> retrieve_game -> evaluate_retrieval -> (game_web_search only if the retrieval is not good enough) -> answer`

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import json
from typing import List, Optional, Literal, TypedDict

import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.tooling import tool
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.memory import ShortTermMemory

MODEL = "gpt-4o-mini"

### Tools

Three tools:
- `retrieve_game`: searches the vector DB
- `evaluate_retrieval`: assesses the retrieval (LLM as judge)
- `game_web_search`: if the retrieval is not good enough, searches the web

#### Retrieve Game Tool

In [4]:
# Load the database built in Part 1: same path, same collection name, same embedding function.
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)
print("Loaded collection", repr(collection.name), "with", collection.count(), "games")


@tool
def retrieve_game(query: str) -> List[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Publisher: Company that published the game
    - Genre: Genre of the game
    - Description: Additional details about the game
    - distance: vector distance to the query (lower is closer)
    """
    res = collection.query(
        query_texts=[query],
        n_results=3,
        include=["metadatas", "distances"],
    )
    return [
        {
            "Platform": m["Platform"],
            "Name": m["Name"],
            "YearOfRelease": m["YearOfRelease"],
            "Publisher": m["Publisher"],
            "Genre": m["Genre"],
            "Description": m["Description"],
            "distance": round(d, 3),
        }
        for m, d in zip(res["metadatas"][0], res["distances"][0])
    ]


retrieve_game

Loaded collection 'udaplay' with 15 games


<Tool name=retrieve_game params=['query']>

#### Evaluate Retrieval Tool

In [5]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="True only if the documents contain the facts needed to answer the question")
    confidence: Literal["high", "medium", "low"] = Field(description="How confident the judge is in that verdict")
    description: str = Field(description="Detailed explanation of the verdict, so an action can be taken")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - confidence: high, medium or low confidence in that verdict
    - description: description about the evaluation result
    """
    system = (
        "You are a strict judge of a retrieval system about video games. "
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Mark useful=true ONLY if the documents are about the exact game (and platform, if asked) "
        "in the question AND contain the specific fact requested (release year, platform, publisher, genre...). "
        "Documents about similar or related games do NOT count. Questions about current events, news or "
        "things not in the documents are NOT answerable. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not."
    )
    user = f"Question: {question}\n\nRetrieved documents:\n{json.dumps(retrieved_docs, indent=2)}"
    ai = LLM(model=MODEL, temperature=0).invoke(
        [SystemMessage(content=system), UserMessage(content=user)],
        response_format=EvaluationReport,
    )
    return EvaluationReport.model_validate_json(ai.content).model_dump()


evaluate_retrieval

<Tool name=evaluate_retrieval params=['question', 'retrieved_docs']>

#### Game Web Search Tool

In [6]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def game_web_search(question: str) -> List[dict]:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    Each element contains: title, url, content.
    """
    res = tavily_client.search(query=f"{question} (video game)", max_results=3)
    return [
        {"title": r["title"], "url": r["url"], "content": r["content"]}
        for r in res["results"]
    ]


game_web_search

<Tool name=game_web_search params=['question']>

### Agent

The agent is a **state machine** built with `lib.state_machine`. The tools are pre-defined nodes, so the order is always: retrieve -> evaluate -> web search *only if needed* -> answer. Each node prints what it did, so the process is visible.

- **Conversation state:** every run is saved in `ShortTermMemory` per session; the question/answer history of the session is passed into the next run, so follow-ups like *"which one was the first 3D one?"* are understood.
- **Structured output:** the final answer is a Pydantic object (`GameAnswer`) with the answer, confidence and cited sources.

In [7]:
class Source(BaseModel):
    kind: Literal["local dataset", "web"]
    reference: str = Field(description="Game name + platform for the local dataset, or the page URL for the web")


class GameAnswer(BaseModel):
    answer: str = Field(description="Clear, natural-language answer to the question")
    confidence: Literal["high", "medium", "low"]
    sources: List[Source] = Field(description="Every source the answer relies on")


class UdaPlayState(TypedDict):
    session_id: str
    question: str
    history: List[dict]           # earlier turns of this session: {"question", "answer"}
    standalone_question: str      # the question rewritten so it makes sense without the history
    retrieved_docs: List[dict]
    evaluation: Optional[dict]
    used_web_search: bool
    web_results: List[dict]
    answer: Optional[dict]
    trace: List[str]              # the steps taken, for the report


INSTRUCTIONS = (
    "You are UdaPlay, an AI research assistant for the video game industry. "
    "Answer the user's question using ONLY the information provided (local dataset documents and/or web results). "
    "Combine both when both are provided. If the information doesn't contain the answer, say so. "
    "Cite every source you use: for local documents use the game name and platform with kind 'local dataset'; "
    "for web results use the exact page URL with kind 'web'. Never invent URLs. "
    "Set confidence to reflect how well the provided information supports the answer."
)

In [8]:
class UdaPlayAgent:
    def __init__(self, model: str = MODEL, instructions: str = INSTRUCTIONS):
        self.model = model
        self.instructions = instructions
        self.memory = ShortTermMemory()          # one list of Runs per session
        self.workflow = self._build_workflow()

    # ---- nodes -------------------------------------------------------------------
    def _log(self, state, text):
        print(text)
        return state["trace"] + [text]

    def _understand(self, state: UdaPlayState) -> dict:
        question = state["question"]
        standalone = question
        if state["history"]:
            history = "\n".join(f"User: {h['question']}\nUdaPlay: {h['answer']}" for h in state["history"])
            ai = LLM(model=self.model, temperature=0).invoke([
                SystemMessage(content="Rewrite the last user question so it is fully self-contained, using the "
                                      "conversation for context (resolve 'it', 'that game', 'which one'...). "
                                      "If it is already self-contained return it unchanged. Return only the question."),
                UserMessage(content=f"Conversation:\n{history}\n\nLast question: {question}"),
            ])
            standalone = ai.content.strip()
        trace = self._log(state, f"[1] Question: {question}")
        if standalone != question:
            trace = trace + [f"    (rewritten using session memory: {standalone})"]
            print(trace[-1])
        return {"standalone_question": standalone, "trace": trace}

    def _retrieve(self, state: UdaPlayState) -> dict:
        docs = retrieve_game(state["standalone_question"])
        names = ", ".join(f"{d['Name']} [{d['Platform']}] (d={d['distance']})" for d in docs)
        return {"retrieved_docs": docs, "trace": self._log(state, f"[2] TOOL retrieve_game -> {names}")}

    def _evaluate(self, state: UdaPlayState) -> dict:
        ev = evaluate_retrieval(state["standalone_question"], state["retrieved_docs"])
        trace = self._log(state, f"[3] TOOL evaluate_retrieval -> useful={ev['useful']}, confidence={ev['confidence']}")
        trace = trace + [f"    reason: {ev['description']}"]
        print(trace[-1])
        return {"evaluation": ev, "trace": trace}

    def _route(self, state: UdaPlayState):
        ev = state["evaluation"]
        return "answer" if ev["useful"] and ev["confidence"] != "low" else "web_search"

    def _web_search(self, state: UdaPlayState) -> dict:
        results = game_web_search(state["standalone_question"])
        urls = ", ".join(r["url"] for r in results)
        return {
            "web_results": results,
            "used_web_search": True,
            "trace": self._log(state, f"[4] TOOL game_web_search (internal knowledge not good enough) -> {urls}"),
        }

    def _answer(self, state: UdaPlayState) -> dict:
        context = f"Local dataset documents:\n{json.dumps(state['retrieved_docs'], indent=2)}"
        if state["used_web_search"]:
            context += f"\n\nWeb search results:\n{json.dumps(state['web_results'], indent=2)}"
        ai = LLM(model=self.model, temperature=0).invoke(
            [SystemMessage(content=self.instructions),
             UserMessage(content=f"Question: {state['standalone_question']}\n\n{context}")],
            response_format=GameAnswer,
        )
        answer = GameAnswer.model_validate_json(ai.content).model_dump()
        return {"answer": answer, "trace": self._log(state, "[5] Final answer generated")}

    # ---- workflow ----------------------------------------------------------------
    def _build_workflow(self) -> StateMachine[UdaPlayState]:
        sm = StateMachine[UdaPlayState](UdaPlayState)
        entry = EntryPoint[UdaPlayState]()
        understand = Step[UdaPlayState]("understand", self._understand)
        retrieve = Step[UdaPlayState]("retrieve_game", self._retrieve)
        evaluate = Step[UdaPlayState]("evaluate_retrieval", self._evaluate)
        web = Step[UdaPlayState]("web_search", self._web_search)
        answer = Step[UdaPlayState]("answer", self._answer)
        end = Termination[UdaPlayState]()
        sm.add_steps([entry, understand, retrieve, evaluate, web, answer, end])

        sm.connect(entry, understand)
        sm.connect(understand, retrieve)
        sm.connect(retrieve, evaluate)
        sm.connect(evaluate, [answer, web], self._route)   # good enough -> answer, else -> web
        sm.connect(web, answer)
        sm.connect(answer, end)
        return sm

    # ---- public API --------------------------------------------------------------
    def ask(self, question: str, session_id: str = "default") -> UdaPlayState:
        self.memory.create_session(session_id)
        history = []
        last = self.memory.get_last_object(session_id)
        if last:
            prev = last.get_final_state()
            history = prev["history"] + [{"question": prev["question"], "answer": prev["answer"]["answer"]}]
        state: UdaPlayState = {
            "session_id": session_id, "question": question, "history": history,
            "standalone_question": question, "retrieved_docs": [], "evaluation": None,
            "used_web_search": False, "web_results": [], "answer": None, "trace": [],
        }
        run = self.workflow.run(state)
        self.memory.add(run, session_id)
        return run.get_final_state()

    def reset_session(self, session_id: str = "default"):
        self.memory.reset(session_id)


def report(result: UdaPlayState):
    a = result["answer"]
    print("\n" + "=" * 78)
    print("UDAPLAY REPORT")
    print("=" * 78)
    print(f"Question   : {result['question']}")
    print(f"Answer     : {a['answer']}")
    print(f"Confidence : {a['confidence']}")
    print("Sources    :")
    for s in a["sources"]:
        print(f"   - [{s['kind']}] {s['reference']}")
    print("Process    : retrieve -> evaluate" + (" -> web search" if result["used_web_search"] else "") + " -> answer")
    print("=" * 78 + "\n")

### Run the agent
Questions in the **same session**, so the agent remembers earlier turns. Two are answered from the local dataset; *Mortal Kombat X* is not in the dataset, so the agent falls back to the web.

In [9]:
agent = UdaPlayAgent()
SESSION = "demo-session"

# Question 1: in the local dataset
print("#" * 78, "\n# QUERY 1\n", "#" * 78)
r1 = agent.ask("When was Pokémon Gold and Silver released?", SESSION)
report(r1)

############################################################################## 
# QUERY 1
 ##############################################################################
[StateMachine] Starting: __entry__
[1] Question: When was Pokémon Gold and Silver released?
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Pokémon Gold and Silver [Game Boy Color] (d=0.111), Pokémon Ruby and Sapphire [Game Boy Advance] (d=0.148), Super Mario 64 [Nintendo 64] (d=0.226)
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=True, confidence=high
    reason: The retrieved documents contain information specifically about 'Pokémon Gold and Silver', including the exact release year of 1999, which directly answers the query. The document also specifies the platform (Game Boy Color) and provides additional relevant details such as the publisher and genre, but the key fact requested (release year) is clearly present.
[StateMachine] Executing step: evaluate_retrieval


[5] Final answer generated
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : When was Pokémon Gold and Silver released?
Answer     : Pokémon Gold and Silver were released in 1999.
Confidence : high
Sources    :
   - [local dataset] Pokémon Gold and Silver - Game Boy Color
Process    : retrieve -> evaluate -> answer



In [10]:
# Question 2: in the local dataset
print("#" * 78, "\n# QUERY 2\n", "#" * 78)
r2 = agent.ask("Which one was the first 3D platformer Mario game?", SESSION)
report(r2)

############################################################################## 
# QUERY 2
 ##############################################################################
[StateMachine] Starting: __entry__


[1] Question: Which one was the first 3D platformer Mario game?
    (rewritten using session memory: Which game was the first 3D platformer featuring Mario?)
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Super Mario 64 [Nintendo 64] (d=0.11), Super Mario World [Super Nintendo Entertainment System (SNES)] (d=0.135), Mario Kart 8 Deluxe [Nintendo Switch] (d=0.197)
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=True, confidence=high
    reason: The documents contain information specifically about 'Super Mario 64', which is the first 3D platformer featuring Mario. It provides the necessary details such as the platform (Nintendo 64), the year of release (1996), the publisher (Nintendo), and the genre (Platformer). This directly answers the query regarding the first 3D platformer featuring Mario.
[StateMachine] Executing step: evaluate_retrieval


[5] Final answer generated
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : Which one was the first 3D platformer Mario game?
Answer     : The first 3D platformer featuring Mario is "Super Mario 64," released in 1996 for the Nintendo 64.
Confidence : high
Sources    :
   - [local dataset] Super Mario 64 - Nintendo 64
Process    : retrieve -> evaluate -> answer



In [11]:
# Question 3: NOT in the local dataset -> web search fallback
print("#" * 78, "\n# QUERY 3\n", "#" * 78)
r3 = agent.ask("Was Mortal Kombat X released for PlayStation 5?", SESSION)
report(r3)

############################################################################## 
# QUERY 3
 ##############################################################################
[StateMachine] Starting: __entry__


[1] Question: Was Mortal Kombat X released for PlayStation 5?
    (rewritten using session memory: Was the video game Mortal Kombat X released for the PlayStation 5?)
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Marvel's Spider-Man 2 [PlayStation 5] (d=0.209), Gran Turismo 5 [PlayStation 3] (d=0.218), Halo Infinite [Xbox Series X|S] (d=0.221)
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=False, confidence=high
    reason: The retrieved documents do not contain any information about the game Mortal Kombat X, nor do they mention its release on the PlayStation 5. The documents provided are about different games (Marvel's Spider-Man 2, Gran Turismo 5, and Halo Infinite) that are unrelated to Mortal Kombat X. Therefore, they do not fulfill the requirement to answer the query regarding Mortal Kombat X's availability on the PlayStation 5.
[StateMachine] Executing step: evaluate_retrieval


[4] TOOL game_web_search (internal knowledge not good enough) -> https://www.youtube.com/watch?v=tqsw711ZuAk, https://en.wikipedia.org/wiki/Mortal_Kombat_X, https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
[StateMachine] Executing step: web_search


[5] Final answer generated
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : Was Mortal Kombat X released for PlayStation 5?
Answer     : Mortal Kombat X was not originally released for the PlayStation 5, but it is playable on the PS5 through backward compatibility. Some features available on the PS4 version may be absent when played on PS5.
Confidence : high
Sources    :
   - [web] https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
Process    : retrieve -> evaluate -> web search -> answer



#### More queries from the project scenario

In [12]:
print("#" * 78, "\n# QUERY 4\n", "#" * 78)
r4 = agent.ask("Who published Grand Theft Auto: San Andreas?", SESSION)
report(r4)

print("#" * 78, "\n# QUERY 5 (current events: needs the web)\n", "#" * 78)
r5 = agent.ask("What is Rockstar Games working on right now?", SESSION)
report(r5)

############################################################################## 
# QUERY 4
 ##############################################################################
[StateMachine] Starting: __entry__


[1] Question: Who published Grand Theft Auto: San Andreas?
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Grand Theft Auto: San Andreas [PlayStation 2] (d=0.118), Gran Turismo [PlayStation 1] (d=0.201), Gran Turismo 5 [PlayStation 3] (d=0.204)
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=True, confidence=high
    reason: The retrieved documents contain information specifically about 'Grand Theft Auto: San Andreas', including the publisher, which is 'Rockstar Games'. The document provides the exact game name, platform (PlayStation 2), release year (2004), and the requested fact about the publisher. Since the question asks specifically for the publisher of 'Grand Theft Auto: San Andreas', and the relevant document provides this information directly, it meets the criteria for being marked as useful.
[StateMachine] Executing step: evaluate_retrieval


[5] Final answer generated
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : Who published Grand Theft Auto: San Andreas?
Answer     : Grand Theft Auto: San Andreas was published by Rockstar Games.
Confidence : high
Sources    :
   - [local dataset] Grand Theft Auto: San Andreas, PlayStation 2
Process    : retrieve -> evaluate -> answer

############################################################################## 
# QUERY 5 (current events: needs the web)
 ##############################################################################
[StateMachine] Starting: __entry__


[1] Question: What is Rockstar Games working on right now?
    (rewritten using session memory: What is Rockstar Games currently working on?)
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Grand Theft Auto: San Andreas [PlayStation 2] (d=0.208), Gran Turismo 5 [PlayStation 3] (d=0.219), Halo Infinite [Xbox Series X|S] (d=0.233)
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=False, confidence=high
    reason: The retrieved documents do not contain any information about what Rockstar Games is currently working on. They only provide details about past games, specifically 'Grand Theft Auto: San Andreas' for PlayStation 2, which is not relevant to the current projects of Rockstar Games. The other two documents are about unrelated games ('Gran Turismo 5' and 'Halo Infinite') and do not pertain to Rockstar Games at all. Therefore, the documents do not answer the query.
[StateMachine] Executing step: evaluate_retrieval


[4] TOOL game_web_search (internal knowledge not good enough) -> https://www.youtube.com/watch?v=HeJZF23l4Ic, https://en.wikipedia.org/wiki/Rockstar_Games, https://en.wikipedia.org/wiki/List_of_video_games_published_by_Rockstar_Games
[StateMachine] Executing step: web_search


[5] Final answer generated
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : What is Rockstar Games working on right now?
Answer     : Rockstar Games is currently working on a new game in the Max Payne series, which is a remake. This project is in full production, meaning that the core assets, levels, and systems are being developed.
Confidence : high
Sources    :
   - [web] https://www.youtube.com/watch?v=HeJZF23l4Ic
   - [web] https://en.wikipedia.org/wiki/Rockstar_Games
Process    : retrieve -> evaluate -> web search -> answer



#### Conversation state check
A follow-up with no game name: the agent resolves it from the session memory (the previous question was about Rockstar).

In [13]:
print("#" * 78, "\n# FOLLOW-UP\n", "#" * 78)
r6 = agent.ask("And which of their games in your dataset came out on PlayStation 2?", SESSION)
report(r6)
print("Runs stored in this session:", len(agent.memory.get_all_objects(SESSION)))

############################################################################## 
# FOLLOW-UP
 ##############################################################################
[StateMachine] Starting: __entry__


[1] Question: And which of their games in your dataset came out on PlayStation 2?
    (rewritten using session memory: Which games published by Rockstar Games in your dataset were released on the PlayStation 2?)
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Grand Theft Auto: San Andreas [PlayStation 2] (d=0.183), Gran Turismo [PlayStation 1] (d=0.191), Gran Turismo 5 [PlayStation 3] (d=0.193)
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=True, confidence=high
    reason: The retrieved documents contain information about 'Grand Theft Auto: San Andreas', which is a game published by Rockstar Games specifically for the PlayStation 2. It includes the requested details such as the platform (PlayStation 2), the publisher (Rockstar Games), and the year of release (2004). The other documents do not pertain to the query as they are about different games and platforms. Therefore, the documents are sufficient to answer the question.
[StateMachine] Executing step: evaluate_retrieval


[5] Final answer generated
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : And which of their games in your dataset came out on PlayStation 2?
Answer     : The only game published by Rockstar Games that was released on the PlayStation 2 is "Grand Theft Auto: San Andreas," which was released in 2004.
Confidence : high
Sources    :
   - [local dataset] Grand Theft Auto: San Andreas - PlayStation 2
Process    : retrieve -> evaluate -> answer

Runs stored in this session: 6


### (Optional) Advanced - Stand Out

**1. Structured output.** Every answer is already a JSON-friendly object (`GameAnswer`), ready for integration:

In [14]:
print(json.dumps(r3["answer"], indent=2, ensure_ascii=False))

{
  "answer": "Mortal Kombat X was not originally released for the PlayStation 5, but it is playable on the PS5 through backward compatibility. Some features available on the PS4 version may be absent when played on PS5.",
  "confidence": "high",
  "sources": [
    {
      "kind": "web",
      "reference": "https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved"
    }
  ]
}


**2. Persistent long-term memory.** After an answer that needed the web, the agent stores what it learned (with the source URL) in a separate persistent ChromaDB collection, `udaplay_learned`. In a *new* agent and session, that knowledge is found in memory without another web search. The main `udaplay` collection is never modified.

In [15]:
import hashlib


class LongTermMemoryAgent(UdaPlayAgent):
    """UdaPlayAgent that also remembers what it learned from the web, across sessions."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.learned = chroma_client.get_or_create_collection("udaplay_learned", embedding_function=embedding_fn)

    def _retrieve(self, state):
        out = super()._retrieve(state)
        if self.learned.count():
            res = self.learned.query(query_texts=[state["standalone_question"]], n_results=1,
                                     include=["documents", "metadatas", "distances"])
            for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
                if dist < 0.35:   # only close matches
                    out["retrieved_docs"] = [{
                        "Name": "Learned from an earlier web search", "Platform": "n/a",
                        "Description": f"Q: {meta['question']} | A: {doc}", "source_urls": meta["urls"], "distance": round(dist, 3),
                        "note": "Cite this as kind 'web' using the URLs in source_urls, not as a local dataset document.",
                    }] + out["retrieved_docs"]
                    out["trace"] = out["trace"] + [f"    long-term memory hit (d={dist:.3f}), sources: {meta['urls']}"]
                    print(out["trace"][-1])
        return out

    def _answer(self, state):
        out = super()._answer(state)
        if state["used_web_search"] and out["answer"]["confidence"] != "low":
            urls = [s["reference"] for s in out["answer"]["sources"] if s["kind"] == "web"]
            self.learned.upsert(
                ids=[hashlib.md5(state["standalone_question"].encode()).hexdigest()],
                documents=[out["answer"]["answer"]],
                metadatas=[{"question": state["standalone_question"], "urls": ", ".join(urls)}],
            )
            out["trace"] = out["trace"] + ["    stored in long-term memory"]
            print(out["trace"][-1])
        return out


# Start the demo from an empty memory so the result is reproducible
if "udaplay_learned" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("udaplay_learned")

print("--- Agent A, session 1: needs the web, then learns ---")
agent_a = LongTermMemoryAgent()
report(agent_a.ask("Was Mortal Kombat X released for PlayStation 5?", "s1"))

--- Agent A, session 1: needs the web, then learns ---
[StateMachine] Starting: __entry__
[1] Question: Was Mortal Kombat X released for PlayStation 5?
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Marvel's Spider-Man 2 [PlayStation 5] (d=0.209), Gran Turismo 5 [PlayStation 3] (d=0.224), Marvel's Spider-Man [PlayStation 4] (d=0.225)
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=False, confidence=high
    reason: The retrieved documents do not contain any information about Mortal Kombat X, nor do they mention its release for PlayStation 5. Instead, the documents focus on other games such as Marvel's Spider-Man 2, Gran Turismo 5, and Marvel's Spider-Man, which are unrelated to Mortal Kombat X. Since the question specifically asks about Mortal Kombat X and its availability on PlayStation 5, and the documents do not address this game at all, the answer is not useful.
[StateMachine] Executing step: evaluate_retrieval


[4] TOOL game_web_search (internal knowledge not good enough) -> https://www.youtube.com/watch?v=tqsw711ZuAk, https://en.wikipedia.org/wiki/Mortal_Kombat_X, https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
[StateMachine] Executing step: web_search


[5] Final answer generated


    stored in long-term memory
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : Was Mortal Kombat X released for PlayStation 5?
Answer     : Mortal Kombat X was not originally released for PlayStation 5, but it is playable on the PS5 through backward compatibility. Some features available on PS4 may be absent when played on PS5.
Confidence : high
Sources    :
   - [web] https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
Process    : retrieve -> evaluate -> web search -> answer



In [16]:
print("--- Agent B (brand-new agent and session): answered from long-term memory ---")
agent_b = LongTermMemoryAgent()
report(agent_b.ask("Was Mortal Kombat X released for PlayStation 5?", "s2"))

--- Agent B (brand-new agent and session): answered from long-term memory ---
[StateMachine] Starting: __entry__
[1] Question: Was Mortal Kombat X released for PlayStation 5?
[StateMachine] Executing step: understand


[2] TOOL retrieve_game -> Marvel's Spider-Man 2 [PlayStation 5] (d=0.209), Gran Turismo 5 [PlayStation 3] (d=0.224), Marvel's Spider-Man [PlayStation 4] (d=0.225)


    long-term memory hit (d=0.097), sources: https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
[StateMachine] Executing step: retrieve_game


[3] TOOL evaluate_retrieval -> useful=True, confidence=high
    reason: The retrieved document provides a clear answer to the question regarding Mortal Kombat X and its availability on PlayStation 5. It states that Mortal Kombat X was not originally released for PlayStation 5 but is playable on the PS5 through backward compatibility. This directly addresses the query about whether the game was released for that platform, fulfilling the requirement for the specific fact requested.
[StateMachine] Executing step: evaluate_retrieval


[5] Final answer generated
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__

UDAPLAY REPORT
Question   : Was Mortal Kombat X released for PlayStation 5?
Answer     : Mortal Kombat X was not originally released for PlayStation 5, but it is playable on the PS5 through backward compatibility. Some features available on PS4 may be absent when played on PS5.
Confidence : high
Sources    :
   - [web] https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved
Process    : retrieve -> evaluate -> answer

